In [ ]:
#****** Automated analysis of coating images ******
#The boundaries should be aligned with the y-axis:

# Boundary
# Coated region   |y
# Boundary        |___x

#The coating height with the z-axis.
#The code automatically loads all excel files, assuming that columns 1, 2 and 3 corresponds to the x, y and z axes, respectively.
#The reference baselime can be established through one of the following two methods:
#reference = 1: baseline is the fitting plane between the boundaries of the coated region.
#reference = 2: baseline is the fitting polynomial of the uncoated borders of the image.

reference = 1

#The results are saved inside the folder with the same name as the file.
#All collected in folder All_Results, which is zipped and can be downloaded.
#To unzip them all, use the following command in the terminal within the All_Results folder:
#for file in *.zip; do
#    # Skip if no .zip files exist
#    [ -e "$file" ] || { echo "No zip files found."; exit 0; }
#    dirname="${file%.zip}"
#   mkdir -p "$dirname"

#   echo "Unzipping $file into $dirname/"
#    unzip -q "$file" -d "$dirname"
#done


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from sklearn.cluster import DBSCAN
from scipy.interpolate import griddata
import pandas as pd
import os
from skimage.filters import threshold_otsu
from scipy.spatial import ConvexHull
from scipy.spatial import Delaunay
from mpl_toolkits.mplot3d import Axes3D
import logging
import shutil
from PIL import Image


# Define PATHS by listing .xlsx or .xls files in /content/
PATHS = []
for filename in os.listdir('/content/'):
    if filename.endswith('.xlsx') or filename.endswith('.xls'):
        PATHS.append(filename)
#If you refer manual control, uncomment below and put the files names.

folders = []
toflag = []
for current_file_name in PATHS:
    FILE_PATH = current_file_name    # path to your three-column x y z file
    # Extract the base name of the file without its extension
    folder_name = os.path.splitext(os.path.basename(FILE_PATH))[0]
    DIRECTORY = folder_name
    folders.append(DIRECTORY)

    # Check if the directory already exists
    if not os.path.exists(DIRECTORY):
        os.makedirs(DIRECTORY)

    ### START COMPUTING ####

    # 1) Load data
    df = pd.read_excel(FILE_PATH)
    df.columns = df.columns.str.strip().str.lower()
    x = df['x'].values
    y = df['y'].values
    z = df['z'].values
    grid = x[1]-x[0]
    #data = np.loadtxt(FILE_PATH)
    #x, y, z = data[:,0], data[:,1], data[:,2]
    N = len(z)
    ymax = max(y); ymin = min(y) # Fixed: ymin was incorrectly set to min(x)
    xmax = max(x); xmin = min(x)
    print(f"FILE: {FILE_PATH}")
    print(f"{N} data points\n")

    # Create a grid over the x-y domain
    xi = np.linspace(min(x), xmax, 100)
    yi = np.linspace(min(y), ymax, 100)
    xi, yi = np.meshgrid(xi, yi)
    # Interpolate z values onto the grid
    zi = griddata((x, y), z, (xi, yi), method='cubic')

    # Production of 8-bit image and Otsu threshold
    # Normalize zi to 0-255 for 8-bit image production
    zi_normalized = ((zi - np.nanmin(zi)) / (np.nanmax(zi) - np.nanmin(zi)) * 255).astype(np.uint8)

    # Save the 8-bit image
    img = Image.fromarray(zi_normalized, mode='L') # 'L' for 8-bit grayscale
    img.save(f"{DIRECTORY}/8_bit_zi_image.tiff")

    # Determine Otsu threshold on the 8-bit image
    otsu_threshold_zi = threshold_otsu(zi_normalized)
    print(f"Otsu's threshold for 8-bit interpolated image (zi): {otsu_threshold_zi}")
    exit()

    report = open(f"{DIRECTORY}/report.txt", "w")
    report.write(f"Data point in this file:    {N}\n")
    report.write(f"Method for defining baseline:\n")
    if reference == 0:report.write(f"Average height of uncoated region\n\n")
    if reference == 1:report.write(f"Plane between boundaries of coated-uncoated regions\n\n")
    if reference == 2:report.write(f"Paraboloid fitting the uncoated region\n\n")

    #Perform filtering to determine coating area
    otsu_threshold_1 = threshold_otsu(z)
    print(f"Otsu's threshold for coating region definition: {otsu_threshold_1}")

    mask_above_threshold = z > otsu_threshold_1
    x_filtered = x[mask_above_threshold]
    y_filtered = y[mask_above_threshold]
    z_filtered = z[mask_above_threshold]
    coords = np.column_stack((x_filtered, y_filtered))


    #Determine coating region boundaries
    logging.getLogger('matplotlib.font_manager').disabled = True

    mask_left = x < xmax /4.0
    unique_y_values = np.unique(y[mask_left])
    y_left = y[mask_left]
    x_left = x[mask_left] # Corrected: should use x for x_left
    z_left = z[mask_left] # Filter z for the left region as well

    # Calculate the average z for each unique y in the left region
    average_z_by_y = []
    for y_val in unique_y_values:
        mask_for_y_val = (y_left == y_val)
        avg_z = np.mean(z_left[mask_for_y_val]) # Use the filtered z_left array
        average_z_by_y.append(avg_z)


    # Find local minima for overall average_z_by_y (existing logic)
    local_minima = []
    if len(average_z_by_y) > 1:
        # Check first point
        if average_z_by_y[0] < average_z_by_y[1]:
            local_minima.append({'y': unique_y_values[0], 'z': average_z_by_y[0]})

        # Check points in between
        for i in range(1, len(average_z_by_y) - 1):
            if average_z_by_y[i] < average_z_by_y[i-1] and average_z_by_y[i] < average_z_by_y[i+1]:
                local_minima.append({'y': unique_y_values[i], 'z': average_z_by_y[i]})

        # Check last point
        if len(average_z_by_y) > 1 and average_z_by_y[-1] < average_z_by_y[-2]:
            local_minima.append({'y': unique_y_values[-1], 'z': average_z_by_y[-1]})

    if local_minima:
        first_local_min = local_minima[0]
        last_local_min = local_minima[-1]

        print(f"\nFirst local minimum left: y = {first_local_min['y']:.2f} um, z = {first_local_min['z']:.2f} um")
        print(f"Last local minimum left: y = {last_local_min['y']:.2f} um, z = {last_local_min['z']:.2f} um")
        min_y_left = first_local_min['y']
        max_y_left = last_local_min['y']

    else:
        print("\nNo local minima found for overall data.")

    mask_right = x > xmax /4.0
    unique_y_values = np.unique(y[mask_right])
    y_right = y[mask_right] # Renamed to y_right for clarity
    x_right = x[mask_right] # Corrected: should use x for x_right
    z_right = z[mask_right] # Filter z for the right region as well

    # Calculate the average z for each unique y in the right region
    average_z_by_y = []
    for y_val in unique_y_values:
        mask_for_y_val = (y_right == y_val)
        avg_z = np.mean(z_right[mask_for_y_val]) # Use the filtered z_right array
        average_z_by_y.append(avg_z)


    # Find local minima for overall average_z_by_y (existing logic)
    local_minima = []
    if len(average_z_by_y) > 1:
        # Check first point
        if average_z_by_y[0] < average_z_by_y[1]:
            local_minima.append({'y': unique_y_values[0], 'z': average_z_by_y[0]})

        # Check points in between
        for i in range(1, len(average_z_by_y) - 1):
            if average_z_by_y[i] < average_z_by_y[i-1] and average_z_by_y[i] < average_z_by_y[i+1]:
                local_minima.append({'y': unique_y_values[i], 'z': average_z_by_y[i]})

        # Check last point
        if len(average_z_by_y) > 1 and average_z_by_y[-1] < average_z_by_y[-2]:
            local_minima.append({'y': unique_y_values[-1], 'z': average_z_by_y[-1]})

    if local_minima:
        first_local_min = local_minima[0]
        last_local_min = local_minima[-1]

        print(f"\nFirst local minimum right: y = {first_local_min['y']:.2f} um, z = {first_local_min['z']:.2f} um")
        print(f"Last local minimum right: y = {last_local_min['y']:.2f} um, z = {last_local_min['z']:.2f} um")
        min_y_right = first_local_min['y']
        max_y_right = last_local_min['y']


    else:
        print("\nNo local minima found for overall data.")

    if (min_y_left < 0.01*ymax) | (min_y_right < 0.01*ymax):
      print(f"it was not possible to defined uncoated region boundaries")
      shutil.rmtree(DIRECTORY)
      toflag.append(FILE_PATH)
      continue
    if (max_y_left > 0.99*ymax) | (max_y_right > 0.99*ymax):
      print(f"it was not possible to defined uncoated region boundaries")
      shutil.rmtree(DIRECTORY)
      toflag.append(FILE_PATH)
      continue

    # Define values inside and outside coating region
    y_low_left = min_y_left
    y_low_right = min_y_right
    y_high_left = max_y_left
    y_high_right = max_y_right

    y_min_fit = np.min([y_low_left,y_low_right])
    y_max_fit = np.max([y_high_left,y_high_right])

    # Define the two bounding lines (linear interpolation)
    y_low_line  = y_low_left  + (y_low_right  - y_low_left)  * (x / xmax)
    y_high_line = y_high_left + (y_high_right - y_high_left) * (x / xmax)

    # Boolean mask for points inside the band
    mask_in = (y >= y_low_line) & (y <= y_high_line)

    # Split arrays
    x_in,  y_in,  z_in  = x[mask_in],  y[mask_in],  z[mask_in]
    x_out, y_out, z_out = x[~mask_in], y[~mask_in], z[~mask_in]

    x_in = np.array(x_in); x_out = np.array(x_out)
    y_in = np.array(y_in); y_out = np.array(y_out)
    z_in = np.array(z_in); z_out = np.array(z_out)

    #Plot selected area in uncorrected image
    plt.figure(figsize=(16, 12))
    # Dynamically determine vmin and vmax for better visualization
    if zi.min() is not np.nan and zi.max() is not np.nan:
        vmin_plot = np.nanmin(zi)
        vmax_plot = np.nanmax(zi)
        levels = np.linspace(vmin_plot, vmax_plot, 10)
    else:
        vmin_plot=0.0; vmax_plot=450
        levels = np.arange(vmin_plot, vmax_plot + 50, 50)

    contour = plt.contourf(xi, yi, zi, levels=levels, cmap='plasma')
    plt.colorbar(contour, label='Height (um)')
    plt.plot(x, y_low_line, label="Line Low", color="cyan", linewidth=4)
    plt.plot(x, y_high_line, label="Line High", color="cyan",linewidth=4)
    plt.xlabel('x (um)')
    plt.ylabel('y (um)')
    plt.yticks(range(0, int(ymax), 500))
    plt.title(f'Contour plot of uncorrected data: {FILE_PATH}')
    plt.savefig(f"{DIRECTORY}/Selected_area.tiff", dpi=300)
    #plt.show()




    #Total Area (mm^2)
    Area_tot = (xmax - xmin)*(ymax - ymin)*1e-6
    Area_in = Area_tot*len(x_in)/N
    Area_out = Area_tot - Area_in

    #Average uncorrected height (mm^2)
    avg_height_out = np.mean(z_out)
    avg_height_in = np.mean(z_in)

    report.write(f"Points inside the selected coating region:   {len(z_in)}\n")
    report.write(f"Points outside the selected coating region:   {len(z_out)}\n")
    report.write(f"Avg_out: Average height in UNCORRECTED uncoated region:   {avg_height_out:.2f} um\n")
    report.write(f"Avg_in: Average height in UNCORRECTED coated region:   {avg_height_in:.2f} um\n")


    # Reset lists as they will be repopulated based on reference value (except for reference == 0)
    x_fit_flat = []; y_fit_flat = []; z_fit_flat = []; x_plot = []; y_plot = []; z_plot = []

    # Determine mask for fitting based on the reference method
    if reference == 2:
        # Use points from the uncoated region (x_out, y_out, z_out) that are outside the overall local minima boundaries
        mask_for_fit = ((y_out <= y_min_fit) | (y_out >= y_max_fit))
        x_fit_flat = x_out[mask_for_fit]
        y_fit_flat = y_out[mask_for_fit]
        z_fit_flat = z_out[mask_for_fit]
    elif reference == 1:
        # Use points from the uncoated region (x_out, y_out, z_out) within specific bands near the overall local minima
        # This also fixes the ValueError by using '&' and '|' for element-wise boolean operations
        mask_for_fit = ((y_out <= y_min_fit) & (y_out >= y_min_fit - 100)) | \
                      ((y_out >= y_max_fit) & (y_out <= y_max_fit + 100))
        x_fit_flat = x_out[mask_for_fit]
        y_fit_flat = y_out[mask_for_fit]
        z_fit_flat = z_out[mask_for_fit]

    mask_for_plot = ((y_out <= y_min_fit) | (y_out >= y_max_fit))
    x_plot = x_out[mask_for_plot]
    y_plot = y_out[mask_for_plot]
    z_plot = z_out[mask_for_plot]

    # Convert lists to numpy arrays for plane fitting
    x_fit_arr = np.array(x_fit_flat)
    y_fit_arr = np.array(y_fit_flat)
    z_fit_arr = np.array(z_fit_flat)

    if reference == 1:
      # Fit a plane to x_fit_arr, y_fit_arr, z_fit_arr data (z = ax + by + c)
      A = np.c_[x_fit_arr, y_fit_arr, np.ones(len(x_fit_arr))]

    # Use least squares to find the coefficients (a, b, c)
      coeffs, residuals, rank, s = np.linalg.lstsq(A, z_fit_arr, rcond=None)
    # coeffs, residuals, rank, s = np.linalg.lstsq(A, z_out, rcond=None)
      a, b, c = coeffs

    # Use the plane equation to generate z_plane for x, y
      z_plane = a * x + b * y + c

    if reference == 2:
        A = np.c_[
            x_fit_arr**2,
            y_fit_arr**2,
            x_fit_arr * y_fit_arr,
            x_fit_arr,
            y_fit_arr,
            np.ones(len(x_fit_arr))  # Corrected from len(x_out) to len(x_fit_arr)
        ]

      # Use the plane equation to generate z_plane for x, y
        coeffs, residuals, rank, s = np.linalg.lstsq(A, z_fit_arr, rcond=None)
        a, b, c, d, e, f = coeffs
        z_plane = (
        a * x**2
        + b * y**2
        + c * x * y
        + d * x
        + e * y
        + f
    )
    if reference == 0:
      z_plane = np.ones(len(x))*avg_height_out

    # Create a meshgrid for the plane based on the range of x_fit and y_fit
    # Adjust these ranges if you want the plane to cover a broader area
    x_plane = np.linspace(x.min(), x.max(), 10)
    y_plane = np.linspace(y.min(), y.max(), 10)
    X_plane, Y_plane = np.meshgrid(x_plane, y_plane)
    if reference == 2:
      Z_plane = (
          a * X_plane**2
          + b * Y_plane**2
          + c * X_plane * Y_plane
          + d * X_plane
          + e * Y_plane
          + f
      )
    if reference == 1:
      Z_plane = a * X_plane + b * Y_plane + c

    if reference == 0:
      Z_plane = np.full((len(x_plane), len(y_plane)), avg_height_out)

    # Create the 3D plot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot the scattered x_fit, y_fit, z_fit points
    ax.scatter(x_plot, y_plot, z_plot, c='blue', marker='.', label='uncoated region data', alpha=0.6)

    # Plot the fitted plane
    ax.plot_surface(X_plane, Y_plane, Z_plane, color='red', alpha=0.3, label='Baseline surface')

    # Set labels and title
    ax.set_xlabel('X (um)')
    ax.set_ylabel('Y (um)')
    ax.set_zlabel('Z (um)')
    ax.set_title('Fitted Baseline Surface')
    ax.legend()

    # Manipulate the view angle: (elevation angle in the z plane, azimuth angle in the x-y plane)
    ax.view_init(elev=5, azim=5) # You can change these values (e.g., elev=30, azim=45)
    if len(z_out) > 0:
        ax.set_zlim(top=1.2*max(z_out))
    else:
        ax.set_zlim(top=np.max(z_plot) * 1.2 if len(z_plot) > 0 else 100) # Fallback if z_out is empty

    plt.savefig(f"{DIRECTORY}/Baseline_Surface.tiff", dpi=300)
    plt.tight_layout()
    #plt.show()

    #Correct image
    z_corrected = z - z_plane
    # Interpolate z values onto the grid
    zii = griddata((x, y), z_corrected, (xi, yi), method='cubic')

    plt.figure(figsize=(16, 12))
    vmin=min(z_corrected); vmax=max(z_corrected)
    levels = np.arange(vmin - 50, vmax + 50, 50)
    contour = plt.contourf(xi, yi, zii, levels=levels, cmap='plasma')

    plt.colorbar(contour, label='Height (um)')
    plt.xlabel('x (um)')
    plt.ylabel('y (um)')
    plt.yticks(range(0, int(ymax), 500))
    #plt.grid(axis = 'y')

    plt.title('Corrected Contour Plot')
    plt.savefig(f"{DIRECTORY}/Background_Corrected_Image.tiff", dpi=300)
    #plt.show()

    #Histogram & cumulative distribution of z
    n_bins = 100
    factor = 1; #z_inc = []; z_plane_inc = []; z_filt = []; z_plane_filt = []

    #Correct z_in, z_filtered
    if reference == 1:
        z_plane_inc = a * x_in + b * y_in + c
        z_plane_filt = a * x_filtered + b * y_filtered + c
    if reference == 2:
      z_plane_inc = (
        a * x_in**2
        + b * y_in**2
        + c * x_in * y_in
        + d * x_in
        + e * y_in
        + f
    )
      z_plane_filt = (
        a * x_filtered**2
        + b * y_filtered**2
        + c * x_filtered * y_filtered
        + d * x_filtered
        + e * y_filtered
        + f
    )
    if reference == 0:
        z_plane_inc = np.ones(len(x_in))*avg_height_out
        z_plane_filt = np.ones(len(x_filtered))*avg_height_out

    z_inc = z_in - factor*z_plane_inc
    z_filt = z_filtered - factor*z_plane_filt
    #z_filt = z_filtered - factor*z_plane_filt
    z_filt = z_inc
    z_filt = z_inc
    maxfilt = np.max(z_filt)
    minfilt = np.min(z_filt)
    z_filt = (z_filt-np.min(z_filt))/(maxfilt-minfilt)

    # Apply Otsu's method to the z_filtered values
    otsu_threshold = 0
    if len(z_filt) > 0 and np.std(z_filt) > 0: # Ensure z_filt is not empty and has variance
        masktest = z_filt > 0
        zmean = np.mean(z_filt(masktest))
        masktest = z_filt > zmean
        if np.any(masktest):
            otsu_threshold = threshold_otsu(z_filt[masktest])
        else:
            print("Warning: No points above mean z_filt for Otsu threshold. Defaulting to 0.")
    else:
        print("Warning: z_filt is empty or has no variance. Otsu threshold not applicable. Defaulting to 0.")

    #print(otsu_threshold*(maxfilt-minfilt)+minfilt)
    #otsu_threshold = min((otsu_threshold*(maxfilt-minfilt)+minfilt)*1.15, 200)
    print(f"Otsu's threshold for defining protrusions: {otsu_threshold} um")

    # Compute histogram
    hist, bins = np.histogram(z_inc, bins=n_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    # Find the index of the bin with the highest count
    mode_index = np.argmax(hist)
    mode_bin_center = bin_centers[mode_index]
    median = np.median(z_inc)
    Q1 = np.percentile(z_inc, 25)
    Q3 = np.percentile(z_inc, 75)
    # Compute the Interquartile Range
    IQR = Q3 - Q1

    # Save histogram
    np.savetxt(f"{DIRECTORY}/histo.txt",
    np.column_stack((bin_centers, hist)),
    header="thickness count", comments='')

    # Compute & save cumulative distribution
    cum_counts = np.cumsum(hist)
    cum_dist   = cum_counts / cum_counts[-1]
    np.savetxt(f"{DIRECTORY}/cumulative.txt",
              np.column_stack((bin_centers, cum_dist)),
              header="thickness cumulative_prob", comments='')

    # Plot histogram
    plt.figure(figsize=(6,4))
    plt.bar(bin_centers, hist,
                width=(bins[1]-bins[0]),
                color='C0', edgecolor='black')
    #plt.xlim(0, 600)
    #plt.ylim(0, 35000)
    plt.xlabel("Thickness (um)")
    plt.ylabel("Count")
    #plt.title("Histogram of thicknesses")
    plt.tight_layout()
    plt.savefig(f"{DIRECTORY}/Thickesses_Histogram.tiff", dpi=300)

    x1 = otsu_threshold
    x2 = 0.0
    plt.axvline(x1, color='red', linestyle='--', linewidth=2, label="Protrusion threshold")
    plt.axvline(x2, color='green', linestyle='--', linewidth=2, label="Depression threshold")
    plt.legend()
    #plt.show()

    # Plot cumulative distribution of z
    plt.figure(figsize=(6,4))
    plt.plot(bin_centers, cum_dist,
                marker='o', color='C1')
    plt.axvline(x1, color='red', linestyle='--', label="Protrusion threshold")
    plt.axvline(x2, color='green', linestyle='--', label="Depression threshold")
    #plt.xlim(0, 550)
    plt.ylim(-0.1, 1.1)
    plt.xlabel("Thikness (um)")
    plt.ylabel("Cumulative Probability")
    #plt.title("Cumulative Distribution of thiknesses")
    plt.tight_layout()
    plt.savefig(f"{DIRECTORY}/Cumulative_Histogram.tiff", dpi=300)
    plt.legend()
    #plt.show()

    report.write("\n*** STATISTICS ***\n")
    report.write(f"Otsu: Otsu's threshold for defining protrusions:  {otsu_threshold:.2f} um\n")
    report.write(f"Q1: Q1 (25th percentile):  {Q1:.2f} um\n")
    report.write(f"Q3: Q3 (75th percentile): {Q3:.2f} um\n")
    report.write(f"IQR: Interquartile Range (IQR): {IQR:.2f} um\n")
    report.write(f"Mode: {mode_bin_center:.2f}\n")
    report.write(f"Median: {median:.2f}\n")

    print("\n*** STATISTICS ***")
    print(f"Otsu's threshold for defining protrusions:  {otsu_threshold:.2f} um")
    print(f"Q1 (25th percentile):  {Q1:.2f} um")
    print(f"Q3 (75th percentile): {Q3:.2f} un")
    print(f"IQR: Interquartile Range: {IQR:.2f} um")
    print(f"Mode: {mode_bin_center:.2f}")
    print(f"Median {median:.2f}")

    #Determine protrusions
    mask_above_threshold = z_inc > 1.2*otsu_threshold
    x_prot = x_in[mask_above_threshold]
    y_prot = y_in[mask_above_threshold]
    z_prot = z_inc[mask_above_threshold]

    #Determine Depressions
    mask_dep = z_inc < -20.0  #This prevents taking regions in the boundary as depressions.
    x_dep = x_in[mask_dep]
    y_dep = y_in[mask_dep]
    z_dep = z_inc[mask_dep]

    # Corrected line for element-wise comparison
    mask_norm = (z_inc >= 0) & (z_inc <= otsu_threshold)
    x_norm = x_in[mask_norm]
    y_norm = y_in[mask_norm]
    z_norm = z_inc[mask_norm]

    poor_coating = 10
    # Corrected line for element-wise comparison
    mask_poor = (z_inc >= 0) & (z_inc <= poor_coating)
    x_poor = x_in[mask_poor]
    y_poor = y_in[mask_poor]
    z_poor = z_inc[mask_poor]

    plt.figure(figsize=(6,6))
    color = "blue"
    plt.scatter(x_norm, y_norm, c=[color], s=30)
    color = "red"
    plt.scatter(x_prot, y_prot, c=[color], s=10)
    #color = "green"
    #plt.scatter(x_dep, y_dep, c=[color], s=30)

    plt.xlabel("x (um)")
    plt.ylabel("y (um)")
    plt.savefig(f"{DIRECTORY}/Analyzed_plot.tiff", dpi=300)
    #plt.show()

    #Calculate relevant areas
    N_in = len(x_in)
    Atrue = Area_in*len(x_norm)/N_in
    Adep = Area_in*len(x_dep)/N_in
    Aprot = Area_in*len(x_prot)/N_in
    Apoor = Area_in*len(x_poor)/N_in

    print(f"Total area: {Area_tot} mm^2")
    print(f"Coated region: {Area_in} mm^2, {Area_in/Area_tot*100} %")
    print(f"Uncoated region area: {Area_out} mm^2, {Area_out/Area_tot*100} %")
    print(f"Coated area (without depression and protrusion): {Atrue} mm^2, {Atrue/Area_in*100} % of coated region")
    print(f"Depressed area: {Adep} mm^2, {Adep/Area_in*100} % of coated region")
    print(f"protruding area: {Aprot} mm^2, {Aprot/Area_in*100} % of coated region")
    print(f"Coated + protruding area: {Atrue + Aprot} mm^2, {(Atrue + Aprot)/Area_in*100} % of coated region")
    print("--------------")

    gridx = abs(x[1] - x[0])
    gridy = abs(y[1] - y[0])
    area_el = gridx*gridy

    c_vol = 0
    for zj in z_norm:
      c_vol += zj*area_el
    for zj in z_prot:
      c_vol += zj*area_el

    print(f"Coating volume: {c_vol*1e-9} mm^3")

    report.write("\n\n*** SURFACE ANALYSIS ***\n")

    report.write(f"A_tot: Total image area:  {Area_tot:.2f} mm^2\n")
    report.write(f"A_coat: Coating region area:   {Area_in:.2f} mm^2    {Area_in/Area_tot*100:.3f} % of total area\n\n")
    report.write(f"A_uncoat: Uncoated region area:  {Area_out:.2f} mm^2   {Area_out/Area_tot*100:.3f} % of total area\n")
    report.write(f"A_blue: Uniform coating (without depressions or protrusions):  {Atrue:.2f} mm^2    {Atrue/Area_in*100:.3f} % of coating region\n")
    report.write(f"A_depr: Depressed area:    {Adep:.2f} mm^2    {Adep/Area_in*100:.3f} % of coating region\n")
    report.write(f"A_prot: protruding area:   {Aprot:.2f} mm^2   {Aprot/Area_in*100:.3f} % of coating region\n")
    report.write(f"A_poor: poorly coated area:   {Apoor:.2f} mm^2   {Apoor/Area_in*100:.3f} % of coating region\n")
    report.write(f"Coated + protruding area:    {Atrue + Aprot:.2f} mm^2    {(Atrue + Aprot)/Area_in*100:.3f} % of coating region\n")
    report.write(f"Coat_vol: Coating volume:    {c_vol*1e-9:.2f} mm^3\n")
    report.close()


# Define the base content directory where individual experiment folders are located
content_dir = '/content/'
# Define the path for the consolidated results directory
all_results_dir = os.path.join(content_dir, 'All_Results')

# Create the 'All_Results' directory if it doesn't exist
os.makedirs(all_results_dir, exist_ok=True)
print(f"Directory '{all_results_dir}' ensured to exist.")

# Move each individual experiment folder into the 'All_Results' directory
print(f"Moving individual experiment folders to '{all_results_dir}':")
folders_moved_count = 0
for folder_name in folders:
    source_path = os.path.join(content_dir, folder_name)
    destination_path = os.path.join(all_results_dir, folder_name)
    # Check if the source directory exists before trying to move it
    if os.path.isdir(source_path):
        try:
            shutil.move(source_path, destination_path)
            print(f"  Moved: {folder_name}")
            folders_moved_count += 1
        except Exception as e:
            print(f"  Error moving {folder_name}: {e}")
    else:
        print(f"  Warning: Source directory '{source_path}' does not exist, skipping.")

print(f"Finished moving {folders_moved_count} individual experiment folders.")

# Create a zip archive of the 'All_Results' directory
output_zip_path = os.path.join(content_dir, 'All_Results') # Name of the zip file will be All_Results.zip
shutil.make_archive(output_zip_path, 'zip', all_results_dir)
print(f"Successfully created zip archive: {output_zip_path}.zip")

# Remove the original 'All_Results' directory after zipping
#shutil.rmtree(all_results_dir)
print(f"Successfully removed original directory: {all_results_dir}")

for s in toflag:
  print(f"\nManually check these files: {s}")


Output hidden; open in https://colab.research.google.com to view.